# 04 - Feature Engineering

**Projeto:** Análise do Mercado Imobiliário Português  
**Fase CRISP-DM:** Data Preparation / Feature Engineering  
**Objetivo:** criar variáveis candidatas para análise avançada e modelação, mantendo uma política explícita de prevenção de fuga de informação.  
**Última alteração:** 18/06/2026

Este notebook usa a versão preparada criada na fase 03 e acrescenta features derivadas de área, divisões, amenidades, idade do imóvel, tempo de publicação e localização. A variável alvo continua a ser `price`.

Entradas esperadas:
- `data/processed/portugal_listings_prepared.csv`, quando disponível localmente.
- `data/processed/portugal_listings_prepared.csv.gz`, versão recomendada para GitHub e Kaggle.
- `/kaggle/input/**/portugal_listings_prepared.csv.gz`, quando o ficheiro estiver anexado ao notebook Kaggle.
- Ficheiro comprimido no GitHub, como alternativa.

Saídas esperadas:
- `data/processed/portugal_listings_features.csv`, em ambiente local.
- `data/processed/portugal_listings_features.csv.gz`, versão comprimida recomendada para GitHub.
- `/kaggle/working/portugal_listings_features.csv` e `/kaggle/working/portugal_listings_features.csv.gz`, em ambiente Kaggle.

Nota metodológica:
- `price_m2`, `price_iqr_outlier` e `price_m2_iqr_outlier` não devem ser usados como preditores de `price`, porque usam direta ou indiretamente a variável alvo.


## Política de Features

A fase de feature engineering deve criar variáveis úteis sem antecipar informação que não estaria disponível no momento da previsão.

| Grupo | Exemplos | Decisão |
|---|---|---|
| Target | `price`, `log_price` | Não entram como preditores |
| Leakage direto | `price_m2` | Não entra como preditor de `price` |
| Diagnóstico baseado no alvo | `price_iqr_outlier`, `price_m2_iqr_outlier` | Não entra como preditor |
| Áreas transformadas | `log_total_area`, rácios de área | Candidatas |
| Divisões e casas de banho | rácios entre quartos, divisões e casas de banho | Candidatas |
| Amenidades | estacionamento, garagem, elevador, carregamento elétrico | Candidatas |
| Localização | `district_type`, `city_type` | Candidatas, com encoding posterior |
| Tempo | trimestre, mês cíclico | Candidatas, dependentes da qualidade de `publish_date` |


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

GITHUB_BASE_URL = "https://raw.githubusercontent.com/LuiscnFigueira/portugal-real-estate-market-analysis/main"
PREPARED_FILENAME = "portugal_listings_prepared.csv"
PREPARED_COMPRESSED_FILENAME = "portugal_listings_prepared.csv.gz"
FEATURE_FILENAME = "portugal_listings_features.csv"
FEATURE_COMPRESSED_FILENAME = "portugal_listings_features.csv.gz"


def find_project_root(start=None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        markers = [candidate / "README.md", candidate / "requirements.txt", candidate / "data"]
        if all(marker.exists() for marker in markers):
            return candidate
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    return current


PROJECT_ROOT = find_project_root()
IS_KAGGLE = Path("/kaggle/working").exists()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LOCAL_PREPARED_PATH = PROJECT_ROOT / "data" / "processed" / PREPARED_FILENAME
LOCAL_PREPARED_COMPRESSED_PATH = PROJECT_ROOT / "data" / "processed" / PREPARED_COMPRESSED_FILENAME
KAGGLE_FEATURE_PATH = Path("/kaggle/working") / FEATURE_FILENAME
LOCAL_FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / FEATURE_FILENAME
FEATURE_DATA_PATH = KAGGLE_FEATURE_PATH if IS_KAGGLE else LOCAL_FEATURE_PATH
FEATURE_COMPRESSED_PATH = FEATURE_DATA_PATH.with_name(FEATURE_COMPRESSED_FILENAME)

GITHUB_PREPARED_URL = f"{GITHUB_BASE_URL}/data/processed/{PREPARED_FILENAME}"
GITHUB_PREPARED_COMPRESSED_URL = f"{GITHUB_BASE_URL}/data/processed/{PREPARED_COMPRESSED_FILENAME}"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Ambiente Kaggle: {IS_KAGGLE}")
print(f"Saída features: {FEATURE_DATA_PATH}")
print(f"Saída comprimida: {FEATURE_COMPRESSED_PATH}")


Raiz do projeto: /kaggle/working
Ambiente Kaggle: True
Saída features: /kaggle/working/portugal_listings_features.csv
Saída comprimida: /kaggle/working/portugal_listings_features.csv.gz


## Funções de Feature Engineering

Sempre que o módulo `src.features.build_features` estiver disponível, o notebook usa as funções reutilizáveis do projeto. Em ambiente Kaggle isolado, são criadas funções equivalentes de fallback.


In [2]:
try:
    from src.features.build_features import add_modeling_features, modeling_feature_columns
    print("Funções carregadas a partir de src.features.build_features")
except ModuleNotFoundError:
    print("Módulo src não encontrado. A usar funções locais de fallback.")

    def _safe_divide(numerator, denominator):
        result = numerator / denominator.replace(0, np.nan)
        return result.replace([np.inf, -np.inf], np.nan)

    def _log1p_positive(series):
        numeric = pd.to_numeric(series, errors="coerce")
        return np.log1p(numeric.where(numeric >= 0))

    def _boolean_signal(series):
        if pd.api.types.is_bool_dtype(series):
            return series.fillna(False).astype(bool)
        normalized = series.astype("string").str.lower().str.strip()
        mapped = normalized.map({
            "true": True, "false": False, "yes": True, "no": False,
            "sim": True, "não": False, "nao": False, "1": True, "0": False,
        })
        return mapped.fillna(False).astype(bool)

    def add_modeling_features(df):
        features = df.copy()
        if "price" in features.columns:
            features["log_price"] = _log1p_positive(features["price"])
        for col in ["total_area", "living_area", "gross_area", "lot_size", "built_area"]:
            if col in features.columns:
                features[f"log_{col}"] = _log1p_positive(features[col])
        if {"living_area", "total_area"}.issubset(features.columns):
            features["living_total_area_ratio"] = _safe_divide(features["living_area"], features["total_area"])
        if {"gross_area", "total_area"}.issubset(features.columns):
            features["gross_total_area_ratio"] = _safe_divide(features["gross_area"], features["total_area"])
        if {"built_area", "total_area"}.issubset(features.columns):
            features["built_total_area_ratio"] = _safe_divide(features["built_area"], features["total_area"])
        if {"number_of_bedrooms", "total_rooms"}.issubset(features.columns):
            features["bedrooms_per_room"] = _safe_divide(features["number_of_bedrooms"], features["total_rooms"])
        if {"number_of_bathrooms", "number_of_bedrooms"}.issubset(features.columns):
            features["bathrooms_per_bedroom"] = _safe_divide(features["number_of_bathrooms"], features["number_of_bedrooms"])
        if {"total_rooms", "number_of_bathrooms"}.issubset(features.columns):
            features["rooms_per_bathroom"] = _safe_divide(features["total_rooms"], features["number_of_bathrooms"])
        amenity_cols = [col for col in ["has_parking", "garage", "elevator", "electric_cars_charging"] if col in features.columns]
        if amenity_cols:
            amenity_frame = pd.DataFrame(
                {col: _boolean_signal(features[col]).astype("Int64") for col in amenity_cols},
                index=features.index,
            )
            features["amenity_count"] = amenity_frame.sum(axis=1, min_count=1)
        if {"parking", "has_parking", "garage"}.intersection(features.columns):
            parking_signal = pd.Series(False, index=features.index)
            if "parking" in features.columns:
                parking_signal = parking_signal | (pd.to_numeric(features["parking"], errors="coerce") > 0)
            if "has_parking" in features.columns:
                parking_signal = parking_signal | _boolean_signal(features["has_parking"])
            if "garage" in features.columns:
                parking_signal = parking_signal | _boolean_signal(features["garage"])
            features["has_parking_or_garage"] = parking_signal
        missing_cols = [col for col in features.columns if col.endswith("_missing")]
        if missing_cols:
            features["missing_core_feature_count"] = features[missing_cols].sum(axis=1)
        if "construction_year" in features.columns:
            construction_year = pd.to_numeric(features["construction_year"], errors="coerce")
            features["construction_decade"] = (construction_year // 10 * 10).astype("Int64")
        if "property_age" in features.columns:
            features["property_age_group"] = pd.cut(
                pd.to_numeric(features["property_age"], errors="coerce"),
                bins=[-1, 5, 15, 30, 60, np.inf],
                labels=["0-5", "6-15", "16-30", "31-60", "60+"],
            ).astype("string")
        if "publish_month" in features.columns:
            publish_month = pd.to_numeric(features["publish_month"], errors="coerce")
            features["publish_quarter"] = (((publish_month - 1) // 3) + 1).astype("Int64")
            features["publish_month_sin"] = np.sin(2 * np.pi * publish_month / 12)
            features["publish_month_cos"] = np.cos(2 * np.pi * publish_month / 12)
        if {"district", "type"}.issubset(features.columns):
            features["district_type"] = features["district"].astype("string").fillna("unknown") + "__" + features["type"].astype("string").fillna("unknown")
        if {"city", "type"}.issubset(features.columns):
            features["city_type"] = features["city"].astype("string").fillna("unknown") + "__" + features["type"].astype("string").fillna("unknown")
        return features

    def modeling_feature_columns(df):
        excluded = {"price", "log_price", "price_m2", "price_iqr_outlier", "price_m2_iqr_outlier"}
        excluded.update(col for col in df.columns if col.endswith("_iqr_outlier"))
        return [col for col in df.columns if col not in excluded]


Módulo src não encontrado. A usar funções locais de fallback.


## Carregamento do Dataset Preparado

O notebook usa preferencialmente a versão comprimida `.csv.gz`, porque é a versão adequada para GitHub e Kaggle. O pandas lê diretamente ficheiros `.csv.gz` sem passos adicionais.


In [3]:
def kaggle_candidates(filename: str) -> list[Path]:
    base = Path("/kaggle/input")
    if not base.exists():
        return []
    return sorted(base.glob(f"**/{filename}"))


def read_first_available_csv(sources):
    errors = []
    for source in sources:
        if isinstance(source, Path) and not source.exists():
            continue
        try:
            return pd.read_csv(source, low_memory=False), source
        except Exception as exc:
            errors.append(f"{source}: {exc}")
    raise FileNotFoundError(
        "Não foi possível carregar o dataset preparado. Fontes testadas:"
        + "\n"
        + "\n".join(map(str, sources + errors))
    )


sources = [
    LOCAL_PREPARED_COMPRESSED_PATH,
    LOCAL_PREPARED_PATH,
    *kaggle_candidates(PREPARED_COMPRESSED_FILENAME),
    *kaggle_candidates(PREPARED_FILENAME),
    GITHUB_PREPARED_COMPRESSED_URL,
    GITHUB_PREPARED_URL,
]

df_prepared, data_source = read_first_available_csv(sources)

if "publish_date" in df_prepared.columns:
    df_prepared["publish_date"] = pd.to_datetime(df_prepared["publish_date"], errors="coerce")

print(f"Fonte usada: {data_source}")
print(f"Dataset preparado: {df_prepared.shape[0]:,} linhas e {df_prepared.shape[1]:,} colunas")
display(df_prepared.head())


Fonte usada: https://raw.githubusercontent.com/LuiscnFigueira/portugal-real-estate-market-analysis/main/data/processed/portugal_listings_prepared.csv.gz
Dataset preparado: 126,242 linhas e 49 colunas


,price,district,city,town,type,energy_certificate,gross_area,total_area,parking,has_parking,floor,construction_year,energy_efficiency_level,publish_date,garage,elevator,electric_cars_charging,total_rooms,number_of_bedrooms,number_of_wc,conservation_status,living_area,lot_size,built_area,number_of_bathrooms,price_m2,property_age,publish_year,publish_month,gross_area_missing,living_area_missing,lot_size_missing,built_area_missing,construction_year_missing,total_rooms_missing,number_of_bedrooms_missing,number_of_wc_missing,number_of_bathrooms_missing,energy_certificate_missing,conservation_status_missing,price_iqr_outlier,gross_area_iqr_outlier,total_area_iqr_outlier,living_area_iqr_outlier,lot_size_iqr_outlier,built_area_iqr_outlier,total_rooms_iqr_outlier,number_of_bathrooms_iqr_outlier,price_m2_iqr_outlier
0,"780,000.0000",Vila Real,Valpaços,Carrazedo de Montenegro e Curros,Farm,NC,200.0000,"552,450.0000",0.0000,False,NaN,NaN,NaN,NaT,NaN,False,NaN,NaN,NaN,NaN,NaN,120.0000,NaN,NaN,0.0000,1.4119,NaN,NaN,NaN,False,False,True,True,True,True,True,True,False,False,True,False,False,True,False,False,False,False,False,False
1,"223,000.0000",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,81.0000,1.0000,True,Ground Floor,NaN,NaN,NaT,NaN,True,NaN,2.0000,NaN,NaN,NaN,81.0000,NaN,NaN,2.0000,"2,753.0864",NaN,NaN,NaN,True,False,True,True,True,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False
2,"228,000.0000",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,108.0000,1.0000,True,Ground Floor,NaN,NaN,NaT,NaN,True,NaN,2.0000,NaN,NaN,NaN,108.0000,NaN,NaN,2.0000,"2,111.1111",NaN,NaN,NaN,True,False,True,True,True,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False
3,"250,000.0000",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0000,1.0000,True,1st Floor,NaN,NaN,NaT,NaN,True,NaN,2.0000,NaN,NaN,NaN,114.0000,NaN,NaN,0.0000,"2,192.9825",NaN,NaN,NaN,True,False,True,True,True,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False
4,"250,000.0000",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0000,1.0000,True,2nd Floor,NaN,NaN,NaT,NaN,True,NaN,2.0000,NaN,NaN,NaN,114.0000,NaN,NaN,2.0000,"2,192.9825",NaN,NaN,NaN,True,False,True,True,True,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False


## Criação de Features

As novas variáveis procuram representar relações relevantes sem alterar a variável alvo. A transformação `log_price` é criada para futura avaliação/modelação, mas é tratada como target transformado, não como preditor.


In [4]:
df_features = add_modeling_features(df_prepared)
feature_columns = modeling_feature_columns(df_features)

created_columns = [col for col in df_features.columns if col not in df_prepared.columns]

summary = pd.DataFrame(
    {
        "indicador": [
            "linhas",
            "colunas_entrada",
            "colunas_saida",
            "features_criadas",
            "features_candidatas_modelacao",
        ],
        "valor": [
            df_features.shape[0],
            df_prepared.shape[1],
            df_features.shape[1],
            len(created_columns),
            len(feature_columns),
        ],
    }
)

display(summary)
display(pd.DataFrame({"feature_criada": created_columns}))


/tmp/ipykernel_16/1170336213.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return mapped.fillna(False).astype(bool)
/tmp/ipykernel_16/1170336213.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return mapped.fillna(False).astype(bool)
/tmp/ipykernel_16/1170336213.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return mappe

,indicador,valor
0,linhas,126242
1,colunas_entrada,49
2,colunas_saida,71
3,features_criadas,22
4,features_candidatas_modelacao,59


,feature_criada
0,log_price
1,log_total_area
2,log_living_area
3,log_gross_area
4,log_lot_size
5,log_built_area
6,living_total_area_ratio
7,gross_total_area_ratio
8,built_total_area_ratio
9,bedrooms_per_room


## Auditoria das Features Criadas

Esta auditoria verifica completude, tipo de dados e exemplos das variáveis criadas. Valores em falta em features derivadas podem ser esperados quando as variáveis originais também estão em falta.


In [5]:
created_audit = (
    pd.DataFrame(
        {
            "feature": created_columns,
            "dtype": [str(df_features[col].dtype) for col in created_columns],
            "missing_count": [int(df_features[col].isna().sum()) for col in created_columns],
            "missing_share": [float(df_features[col].isna().mean()) for col in created_columns],
            "unique_values": [int(df_features[col].nunique(dropna=True)) for col in created_columns],
        }
    )
    .sort_values(["missing_share", "feature"], ascending=[False, True])
    .reset_index(drop=True)
)

display(created_audit)


,feature,dtype,missing_count,missing_share,unique_values
0,bedrooms_per_room,float64,105730,0.8375,155
1,built_total_area_ratio,float64,102232,0.8098,11156
2,log_built_area,float64,101711,0.8057,7290
3,gross_total_area_ratio,float64,100985,0.7999,12569
4,log_gross_area,float64,100574,0.7967,2264
5,publish_month_cos,float64,98770,0.7824,11
6,publish_month_sin,float64,98770,0.7824,11
7,publish_quarter,Int64,98770,0.7824,4
8,log_lot_size,float64,90113,0.7138,6770
9,bathrooms_per_bedroom,float64,86922,0.6885,145


## Controlo de Leakage

As features candidatas para modelação excluem a variável alvo, transformações da variável alvo e flags de outliers. Mesmo quando uma flag parece útil, se foi calculada com toda a distribuição antes da separação treino/teste, deve ser tratada com cautela.


In [6]:
excluded_columns = [col for col in df_features.columns if col not in feature_columns]
policy_table = pd.DataFrame(
    {
        "grupo": ["target_ou_leakage", "candidatas_modelacao"],
        "numero_colunas": [len(excluded_columns), len(feature_columns)],
        "exemplos": [", ".join(excluded_columns[:12]), ", ".join(feature_columns[:12])],
    }
)

display(policy_table)


,grupo,numero_colunas,exemplos
0,target_ou_leakage,12,"price, price_m2, price_iqr_outlier, gross_area..."
1,candidatas_modelacao,59,"district, city, town, type, energy_certificate..."


## Gravação do Dataset de Features

São criados dois ficheiros:

- `portugal_listings_features.csv`, útil para inspeção local;
- `portugal_listings_features.csv.gz`, versão comprimida recomendada para GitHub.

No Kaggle, estes ficheiros ficam em `/kaggle/working/`.


In [7]:
FEATURE_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_features.to_csv(FEATURE_DATA_PATH, index=False)
df_features.to_csv(FEATURE_COMPRESSED_PATH, index=False, compression="gzip")

csv_size_mb = FEATURE_DATA_PATH.stat().st_size / (1024 ** 2)
gz_size_mb = FEATURE_COMPRESSED_PATH.stat().st_size / (1024 ** 2)

print(f"Dataset de features guardado em: {FEATURE_DATA_PATH}")
print(f"Versão comprimida guardada em: {FEATURE_COMPRESSED_PATH}")
print(f"Linhas: {df_features.shape[0]:,}")
print(f"Colunas: {df_features.shape[1]:,}")
print(f"Tamanho CSV: {csv_size_mb:.1f} MB")
print(f"Tamanho CSV.GZ: {gz_size_mb:.1f} MB")


Dataset de features guardado em: /kaggle/working/portugal_listings_features.csv
Versão comprimida guardada em: /kaggle/working/portugal_listings_features.csv.gz
Linhas: 126,242
Colunas: 71
Tamanho CSV: 49.8 MB
Tamanho CSV.GZ: 9.1 MB


## Conclusão

Este notebook criou uma versão enriquecida do dataset, acrescentando features candidatas para análise avançada e modelação futura.

Principais pontos a reter:
- foram criadas transformações logarítmicas de áreas e da variável alvo;
- foram criados rácios entre áreas, divisões, quartos e casas de banho;
- foram criadas variáveis de amenidades, idade, mês/trimestre e combinações geográficas;
- `price_m2` e variáveis derivadas diretamente do preço foram excluídas da lista de preditores candidatos;
- o ficheiro comprimido `.csv.gz` é o artefacto recomendado para GitHub e Kaggle.

Artefactos produzidos:
- `data/processed/portugal_listings_features.csv`;
- `data/processed/portugal_listings_features.csv.gz`.

Próxima etapa:
- avançar para validação estatística e/ou baseline de modelação, usando pipelines que façam encoding e imputação dentro da separação treino/teste.

**Última alteração:** 18/06/2026
